# Evaluate ALS (PyTorch 2 GPU) on lastone test split

In [ ]:

import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:

# Load test interactions
base = Path('/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output')
files = sorted(base.glob('test/*.parquet'))
dfs = [pq.read_table(f).to_pandas() for f in files]
inter = pd.concat(dfs, ignore_index=True)
print('interactions:', len(inter))
print(inter.head())


In [ ]:

# Load ALS factors
kbase = Path('/kaggle/input/notebooks/nguynteo/als-torch-2gpu-v3')
X = np.load(kbase/'als_torch_user_factors.npy')
Y = np.load(kbase/'als_torch_item_factors.npy')
u_ids = np.load(kbase/'als_torch_user_ids.npy')
i_ids = np.load(kbase/'als_torch_item_ids.npy')
meta = json.loads((kbase/'als_torch_meta.json').read_text())
print('X', X.shape, 'Y', Y.shape, 'meta', meta)


In [ ]:

# Map raw ids -> contiguous indices
def to_idx(raw, uniq):
    pos = np.searchsorted(uniq, raw)
    ok = (pos < len(uniq)) & (uniq[pos] == raw)
    out = np.full(len(raw), -1, dtype=np.int64)
    out[ok] = pos[ok]
    return out

inter = inter.copy()
inter['u'] = to_idx(inter['user_idx'].to_numpy(), u_ids)
inter['i'] = to_idx(inter['item_idx'].to_numpy(), i_ids)
valid = (inter['u'] >= 0) & (inter['i'] >= 0)
inter = inter[valid].reset_index(drop=True)
print('valid interactions:', len(inter))


In [ ]:

# RMSE / MAE on (sample of) test interactions
rng = np.random.default_rng(42)
sample_n = min(200000, len(inter))
idx = rng.choice(len(inter), size=sample_n, replace=False)
u = inter['u'].to_numpy()[idx]
i = inter['i'].to_numpy()[idx]
r = inter['rating'].to_numpy()[idx].astype(np.float32)
pred = np.sum(X[u] * Y[i], axis=1)
rmse = float(np.sqrt(np.mean((pred - r)**2)))
mae = float(np.mean(np.abs(pred - r)))
print(f'ALS RMSE: {rmse:.4f}  MAE: {mae:.4f}  (n={sample_n})')


In [ ]:

# Popularity baseline: top items by frequency in test
item_pop = inter['i'].value_counts()
popular_items = item_pop.index.to_numpy()[:10]
print('top popular (contig):', popular_items[:5])


In [ ]:

# Sampled ranking evaluation
# For each sampled user: ground truth item + 99 random negatives
N_USERS = 2000
rng = np.random.default_rng(7)
# pick users with at least one interaction
users_with_gt = inter['u'].unique()
chosen_users = rng.choice(users_with_gt, size=min(N_USERS, len(users_with_gt)), replace=False)

all_items = np.arange(len(i_ids))
def ndcg_at_k(hit_pos, k):
    # hit_pos is 0-based position of ground truth in ranked list, or None
    if hit_pos is None or hit_pos >= k:
        return 0.0
    return 1.0 / np.log2(hit_pos + 2)

hr10_als = 0; ndcg10_als = 0.0; mrr_als = 0.0
hr10_pop = 0; ndcg10_pop = 0.0; mrr_pop = 0.0
count = 0
for u in chosen_users:
    # ground truth item for this user (first)
    gt_rows = inter[inter['u'] == u]
    if len(gt_rows) == 0:
        continue
    gt_item = int(gt_rows['i'].iloc[0])
    # negatives
    negs = rng.choice(all_items, size=99, replace=False)
    negs = negs[negs != gt_item]
    cands = np.concatenate([[gt_item], negs])
    # ALS scores
    scores = X[u] @ Y[cands].T
    order_als = np.argsort(-scores)
    ranked_items = cands[order_als]
    # position of gt
    pos = np.where(ranked_items == gt_item)[0]
    pos = int(pos[0]) if len(pos) else None
    if pos is not None and pos < 10:
        hr10_als += 1
        ndcg10_als += ndcg_at_k(pos, 10)
        mrr_als += 1.0/(pos+1)
    # popularity: rank candidates by global popularity
    # map candidate -> pop count
    pop_score = np.array([item_pop.get(int(x), 0) for x in cands], dtype=float)
    order_pop = np.argsort(-pop_score)
    ranked_pop = cands[order_pop]
    posp = np.where(ranked_pop == gt_item)[0]
    posp = int(posp[0]) if len(posp) else None
    if posp is not None and posp < 10:
        hr10_pop += 1
        ndcg10_pop += ndcg_at_k(posp, 10)
        mrr_pop += 1.0/(posp+1)
    count += 1

print(f'Evaluated users: {count}')
print('Model     HR@10   NDCG@10   MRR')
print(f'ALS       {hr10_als/count:.4f}  {ndcg10_als/count:.4f}  {mrr_als/count:.4f}')
print(f'Popularity{hr10_pop/count:.4f}  {ndcg10_pop/count:.4f}  {mrr_pop/count:.4f}')


In [ ]:

# Save metrics
import json as _json
metrics = {
    'rmse': rmse,
    'mae': mae,
    'hr10_als': hr10_als/count,
    'ndcg10_als': ndcg10_als/count,
    'mrr_als': mrr_als/count,
    'hr10_pop': hr10_pop/count,
    'ndcg10_pop': ndcg10_pop/count,
    'mrr_pop': mrr_pop/count,
    'n_eval_users': count,
}
Path('/kaggle/working/als_eval_metrics.json').write_text(_json.dumps(metrics, indent=2))
print(_json.dumps(metrics, indent=2))


In [ ]:
!ls -lh /kaggle/working